# Tuberculosis Detection with TBX11K Dataset: Multi-Model Comparison

This notebook demonstrates how to use different model architectures for tuberculosis detection using the TBX11K dataset. We'll explore alternatives to the SymFormer architecture while keeping the same dataset and evaluation framework.

## Overview:
1. Set up the TBX11K dataset
2. Implement multiple model architectures for TB detection
3. Train and evaluate each model
4. Compare performance across different architectures
5. Visualize results and detection performance

## 1. Import Required Libraries

In [ ]:
# Basic libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import cv2
from PIL import Image

# PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    retinanet_resnet50_fpn,
    ssdlite320_mobilenet_v3_large,
    maskrcnn_resnet50_fpn
)
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader

# Object detection evaluation metrics
from torchvision.utils import draw_bounding_boxes
from torchvision.ops import box_iou

# For results visualization
import seaborn as sns
from IPython.display import display

# Set up matplotlib for visualization
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Loading and Exploration

Let's load and explore the TBX11K dataset structure. We'll need to set up the dataset with its COCO format annotations and examine the distribution of classes.

In [ ]:
# Define dataset paths - update these paths according to your setup
TBX11K_ROOT = "data/TBX11K"  # Root directory of TBX11K dataset
IMAGES_DIR = os.path.join(TBX11K_ROOT, "images")
ANNOTATIONS_DIR = os.path.join(TBX11K_ROOT, "annotations")
LISTS_DIR = os.path.join(TBX11K_ROOT, "lists")

# Define class names for the TBX11K dataset
class_names = [
    "Healthy", 
    "Sick & Non-TB", 
    "Active TB", 
    "Latent TB", 
    "Active & Latent TB",
    "Uncertain TB"
]

# Print dataset statistics
print("TBX11K Dataset Statistics:")
print("-" * 40)
print("Classes:")
for i, cls in enumerate(class_names):
    print(f"  {i}: {cls}")

# Check if dataset directories exist
if os.path.exists(TBX11K_ROOT):
    print("\nFound TBX11K dataset directory.")
    # List subdirectories in the images folder
    if os.path.exists(IMAGES_DIR):
        image_subdirs = [d for d in os.listdir(IMAGES_DIR) if os.path.isdir(os.path.join(IMAGES_DIR, d))]
        print(f"Image subdirectories: {image_subdirs}")
    else:
        print(f"Images directory not found: {IMAGES_DIR}")
    
    # Check annotation files
    if os.path.exists(ANNOTATIONS_DIR):
        annotation_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.json')]
        print(f"Annotation files: {annotation_files}")
    else:
        print(f"Annotations directory not found: {ANNOTATIONS_DIR}")
else:
    print(f"\nTBX11K dataset not found at {TBX11K_ROOT}. Please download and extract it there.")

In [ ]:
# Custom dataset class for TBX11K
class TBX11KDataset(Dataset):
    def __init__(self, ann_file, img_prefix, img_list=None, transforms=None):
        """
        TBX11K Dataset class for PyTorch
        
        Args:
            ann_file (str): Path to the annotation file
            img_prefix (str): Path to the image directory
            img_list (str, optional): Path to a text file containing image paths to use
            transforms (callable, optional): Transformations to apply to the images
        """
        self.img_prefix = img_prefix
        self.transforms = transforms
        
        # Load annotations
        with open(ann_file, 'r') as f:
            self.coco = json.load(f)
        
        # Load image list if provided
        self.img_ids = []
        if img_list is not None and os.path.exists(img_list):
            with open(img_list, 'r') as f:
                img_paths = [line.strip() for line in f.readlines()]
            
            # Create a mapping from file_name to image_id
            fname_to_id = {img['file_name']: img['id'] for img in self.coco['images']}
            
            # Filter images based on the provided list
            for path in img_paths:
                if path in fname_to_id:
                    self.img_ids.append(fname_to_id[path])
        else:
            # Use all images
            self.img_ids = [img['id'] for img in self.coco['images']]
        
        # Create a mapping from image_id to annotations
        self.img_to_anns = {}
        for ann in self.coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_to_anns:
                self.img_to_anns[img_id] = []
            self.img_to_anns[img_id].append(ann)
        
        # Create a mapping from category_id to category name
        self.cat_ids = {}
        for cat in self.coco['categories']:
            self.cat_ids[cat['id']] = cat['name']
    
    def __len__(self):
        return len(self.img_ids)
    
    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        
        # Find the image info
        img_info = None
        for img in self.coco['images']:
            if img['id'] == img_id:
                img_info = img
                break
        
        # Load image
        img_path = os.path.join(self.img_prefix, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")
        
        # Get annotations for this image
        anns = self.img_to_anns.get(img_id, [])
        
        boxes = []
        labels = []
        areas = []
        iscrowd = []
        
        for ann in anns:
            bbox = ann['bbox']  # [x, y, width, height] format
            # Convert to [x1, y1, x2, y2] format
            bbox = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
            boxes.append(bbox)
            
            # Category ID adjustment (assuming categories start from 1)
            cat_id = ann['category_id'] - 1  # Convert to 0-indexed
            labels.append(cat_id)
            
            areas.append(ann['area'])
            iscrowd.append(ann.get('iscrowd', 0))
        
        # Convert to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        areas = torch.as_tensor(areas, dtype=torch.float32)
        iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)
        
        # Prepare target dictionary
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id]),
            "area": areas,
            "iscrowd": iscrowd
        }
        
        image = F.pil_to_tensor(image)
        image = image.float() / 255.0  # Normalize to [0, 1]
        
        if self.transforms:
            image, target = self.transforms(image, target)
        
        return image, target

# Function to get data loaders
def get_data_loaders(ann_file_train, ann_file_val, img_prefix, img_list_train=None, img_list_val=None, batch_size=4):
    # Define transformations
    def get_transform():
        transforms = []
        return torchvision.transforms.Compose(transforms)
    
    # Create datasets
    dataset_train = TBX11KDataset(
        ann_file=ann_file_train,
        img_prefix=img_prefix,
        img_list=img_list_train,
        transforms=get_transform()
    )
    
    dataset_val = TBX11KDataset(
        ann_file=ann_file_val,
        img_prefix=img_prefix,
        img_list=img_list_val,
        transforms=get_transform()
    )
    
    # Create data loaders
    data_loader_train = DataLoader(
        dataset_train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda batch: tuple(zip(*batch))
    )
    
    data_loader_val = DataLoader(
        dataset_val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda batch: tuple(zip(*batch))
    )
    
    return data_loader_train, data_loader_val

In [ ]:
# Function to visualize dataset samples
def visualize_sample(dataset, idx):
    """Visualize a sample from the dataset with bounding boxes and labels"""
    image, target = dataset[idx]
    
    # Convert image tensor to numpy array for visualization
    image_np = (image * 255).byte().permute(1, 2, 0).numpy()
    
    # Create a copy for drawing
    image_with_boxes = image_np.copy()
    
    # Draw bounding boxes and labels
    for i, (box, label) in enumerate(zip(target['boxes'], target['labels'])):
        # Convert box to integers
        x1, y1, x2, y2 = [int(coord) for coord in box.tolist()]
        
        # Get class name
        class_name = class_names[label.item()]
        
        # Choose a color based on the class
        color = (0, 255, 0) if "TB" in class_name else (255, 0, 0)
        
        # Draw the box
        cv2.rectangle(image_with_boxes, (x1, y1), (x2, y2), color, 2)
        
        # Add label text
        cv2.putText(image_with_boxes, class_name, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Display the image with bounding boxes
    plt.figure(figsize=(10, 8))
    plt.imshow(image_with_boxes)
    plt.title(f"Sample {idx} - {len(target['boxes'])} detections")
    plt.axis('off')
    plt.show()
    
    # Print additional information
    print(f"Image ID: {target['image_id'].item()}")
    print(f"Number of objects: {len(target['boxes'])}")
    for i, (box, label, area) in enumerate(zip(target['boxes'], target['labels'], target['area'])):
        class_name = class_names[label.item()]
        print(f"Object {i+1}: {class_name}, Area: {area.item():.2f}, Box: {box.tolist()}")

In [ ]:
# Initialize datasets (when you have the TBX11K dataset ready)
try:
    # Set paths for training data
    ann_file_train = os.path.join(ANNOTATIONS_DIR, "train.json")
    img_list_train = os.path.join(LISTS_DIR, "all_train.txt")
    
    # Create a training dataset instance
    train_dataset = TBX11KDataset(
        ann_file=ann_file_train,
        img_prefix=IMAGES_DIR,
        img_list=img_list_train
    )
    
    print(f"Successfully loaded training dataset with {len(train_dataset)} samples.")
    
    # Visualize a few samples
    if len(train_dataset) > 0:
        print("\nVisualizing sample images with annotations:")
        for idx in [0, 10, 20]:  # Visualize samples at these indices
            if idx < len(train_dataset):
                visualize_sample(train_dataset, idx)
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure you have downloaded the TBX11K dataset and specified the correct paths.")

## 3. Setup Environment and Configuration

Now, let's set up configurations for different models. We'll use pre-trained models from torchvision's detection package and adapt them for TB detection with the TBX11K dataset.

In [ ]:
# Configuration dictionary for different models
model_configs = {
    "faster_rcnn_resnet50": {
        "name": "Faster R-CNN ResNet-50 FPN",
        "model_fn": lambda num_classes: fasterrcnn_resnet50_fpn(pretrained=True, progress=True, 
                                                               num_classes=num_classes, 
                                                               pretrained_backbone=True),
        "description": "Two-stage detector with a Region Proposal Network and a ResNet-50 backbone"
    },
    "retinanet_resnet50": {
        "name": "RetinaNet ResNet-50 FPN",
        "model_fn": lambda num_classes: retinanet_resnet50_fpn(pretrained=True, progress=True,
                                                             num_classes=num_classes,
                                                             pretrained_backbone=True),
        "description": "One-stage detector with focal loss and a ResNet-50 backbone"
    },
    "ssdlite_mobilenet": {
        "name": "SSDLite MobileNetV3-Large",
        "model_fn": lambda num_classes: ssdlite320_mobilenet_v3_large(pretrained=True, progress=True,
                                                                     num_classes=num_classes,
                                                                     pretrained_backbone=True),
        "description": "Lightweight single-shot detector optimized for mobile devices"
    },
    "mask_rcnn_resnet50": {
        "name": "Mask R-CNN ResNet-50 FPN",
        "model_fn": lambda num_classes: maskrcnn_resnet50_fpn(pretrained=True, progress=True,
                                                           num_classes=num_classes,
                                                           pretrained_backbone=True),
        "description": "Instance segmentation model that also performs detection"
    }
}

# Number of classes in the TBX11K dataset
num_classes = len(class_names)  # 6 classes

# Function to create a model
def create_model(model_key):
    """Create and initialize a detection model from the configuration."""
    if model_key not in model_configs:
        raise ValueError(f"Unknown model key: {model_key}")
    
    config = model_configs[model_key]
    print(f"Creating model: {config['name']}")
    print(f"Description: {config['description']}")
    
    model = config["model_fn"](num_classes)
    model.to(device)
    
    return model, config["name"]

# Print available models
print("Available Models for TB Detection:")
for key, config in model_configs.items():
    print(f"- {key}: {config['name']}")
    print(f"  {config['description']}")

## 4. Custom Model Implementation

Let's implement custom model architecture adaptations. This section shows how to adapt models for the specific needs of TB detection beyond what's available in the SymFormer architecture.

In [ ]:
# Custom Feature Pyramid Network for symmetric feature extraction
class SymmetricFPN(nn.Module):
    def __init__(self, backbone_name='resnet50', pretrained=True):
        super(SymmetricFPN, self).__init__()
        
        # Load the backbone model
        if backbone_name == 'resnet50':
            backbone = torchvision.models.resnet50(pretrained=pretrained)
            # Feature extraction layers
            self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
            self.layer1 = backbone.layer1  # 256 channels
            self.layer2 = backbone.layer2  # 512 channels
            self.layer3 = backbone.layer3  # 1024 channels
            self.layer4 = backbone.layer4  # 2048 channels
            
        elif backbone_name == 'efficientnet_b0':
            backbone = torchvision.models.efficientnet_b0(pretrained=pretrained)
            # Feature extraction stages
            self.layer0 = nn.Sequential(backbone.features[0])
            self.layer1 = nn.Sequential(*backbone.features[1:3])
            self.layer2 = nn.Sequential(*backbone.features[3:5])
            self.layer3 = nn.Sequential(*backbone.features[5:7])
            self.layer4 = nn.Sequential(*backbone.features[7:])
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")
        
        # FPN lateral connections (reduce channel dimensions)
        self.lateral4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.lateral3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.lateral2 = nn.Conv2d(512, 256, kernel_size=1)
        self.lateral1 = nn.Conv2d(256, 256, kernel_size=1)
        
        # FPN output connections
        self.output4 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.output3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.output2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.output1 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        
        # Symmetric feature fusion
        self.sym_fusion = nn.Conv2d(256, 256, kernel_size=3, padding=1)
    
    def forward(self, x):
        # Bottom-up pathway
        c1 = self.layer0(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        
        # Top-down pathway and lateral connections
        p5 = self.lateral4(c5)
        p4 = self.lateral3(c4) + F.interpolate(p5, size=c4.shape[-2:], mode='nearest')
        p3 = self.lateral2(c3) + F.interpolate(p4, size=c3.shape[-2:], mode='nearest')
        p2 = self.lateral1(c2) + F.interpolate(p3, size=c2.shape[-2:], mode='nearest')
        
        # Output connections
        p5 = self.output4(p5)
        p4 = self.output3(p4)
        p3 = self.output2(p3)
        p2 = self.output1(p2)
        
        # Apply symmetric feature fusion (simulate bilateral symmetry in CXR)
        # Flip the feature maps horizontally and fuse with original
        flipped_p3 = torch.flip(p3, [3])  # Flip along width dimension
        sym_p3 = self.sym_fusion(p3 + flipped_p3)
        
        return [p2, p3, p4, p5, sym_p3]

# Custom TB detection model integrating SymmetricFPN with Faster R-CNN
class TBDetector(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super(TBDetector, self).__init__()
        
        # Base detection model (we're using Faster R-CNN here)
        self.detector = fasterrcnn_resnet50_fpn(
            pretrained=pretrained,
            pretrained_backbone=True,
            num_classes=num_classes
        )
        
        # Replace the backbone with our symmetric FPN
        # Note: This is a simplified representation - in practice,
        # you would need to carefully integrate the SymmetricFPN
        # with the detection model architecture
        self.symmetric_fpn = SymmetricFPN(backbone_name='resnet50', pretrained=pretrained)
    
    def forward(self, images, targets=None):
        if self.training and targets is None:
            raise ValueError("In training mode, targets should be passed")
            
        # In practice, you would need to integrate the symmetric FPN
        # properly with the detector's forward pass
        return self.detector(images, targets)
        
# Create the custom TB detector
def create_custom_model():
    model = TBDetector(num_classes=num_classes)
    model.to(device)
    return model, "Custom TB Detector with Symmetric FPN"

# Note: This is a simplified implementation to illustrate the concept.
# A full implementation would require more careful integration of components.
print("Custom TB Detector model defined but not instantiated.")

In [ ]:
# Extended model configurations with more backbone options
def create_extended_model_config():
    """
    Create extended model configurations with more backbone options and architectures
    Returns a dictionary of model configurations
    """
    extended_configs = {
        # Original models
        "faster_rcnn_resnet50": {
            "name": "Faster R-CNN ResNet-50 FPN",
            "model_fn": lambda num_classes: fasterrcnn_resnet50_fpn(
                pretrained=True, 
                progress=True, 
                num_classes=num_classes, 
                pretrained_backbone=True
            ),
            "description": "Two-stage detector with a Region Proposal Network and a ResNet-50 backbone"
        },
        
        # Add ResNet-101 backbone for Faster R-CNN
        "faster_rcnn_resnet101": {
            "name": "Faster R-CNN ResNet-101 FPN",
            "model_fn": lambda num_classes: torchvision.models.detection.fasterrcnn_resnet50_fpn(
                pretrained=True,
                progress=True, 
                num_classes=num_classes,
                pretrained_backbone=True,
                # Replace with ResNet-101 backbone - you would need to implement this
                # This is a placeholder showing the idea
                backbone_name='resnet101'
            ),
            "description": "Two-stage detector with a Region Proposal Network and a ResNet-101 backbone"
        },
        
        # Add EfficientNet backbone
        "retinanet_efficientnet": {
            "name": "RetinaNet EfficientNet-B3",
            "model_fn": lambda num_classes: torchvision.models.detection.retinanet_resnet50_fpn(
                pretrained=True,
                progress=True,
                num_classes=num_classes,
                pretrained_backbone=True
                # In practice, you would replace the backbone with EfficientNet
                # This requires custom implementation
            ),
            "description": "One-stage detector with focal loss and an EfficientNet backbone"
        },
        
        # YOLO-style model 
        "yolo_model": {
            "name": "YOLOv5-style Model",
            "model_fn": lambda num_classes: None,  # Placeholder - requires custom implementation
            "description": "Single-shot detector with grid-based approach similar to YOLOv5"
        },
        
        # FCOS - Fully Convolutional One-Stage Detector
        "fcos_detector": {
            "name": "FCOS Detector",
            "model_fn": lambda num_classes: None,  # Placeholder - requires custom implementation
            "description": "Anchor-free, one-stage detector with center-ness branch"
        },
        
        # Dense object detector
        "densenet_detector": {
            "name": "DenseNet-based Detector",
            "model_fn": lambda num_classes: None,  # Placeholder - requires custom implementation
            "description": "Object detector with DenseNet backbone for feature reuse"
        },
    }
    
    return extended_configs

# Note: The extended models above are placeholders to demonstrate the structure.
# To implement them, you would need to:
#  1. Import or create the actual model architectures
#  2. Replace the placeholder model_fn with actual model creation code
#  3. Make any necessary adjustments for TB detection

print("Extended model configurations defined - requires custom implementation for new architectures")

## 5. Training Pipeline

Let's implement a flexible training pipeline that can work with any of our model architectures.

In [ ]:
# Training utilities
def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10):
    """Train the model for one epoch"""
    model.train()
    metric_logger = {}
    metric_logger['loss'] = 0
    metric_logger['lr'] = optimizer.param_groups[0]["lr"]
    
    for i, (images, targets) in enumerate(data_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        metric_logger['loss'] += losses.item()
        
        if i % print_freq == 0:
            print(f"Epoch: [{epoch}] [{i}/{len(data_loader)}], Loss: {losses.item():.4f}")
    
    # Compute average loss for the epoch
    metric_logger['loss'] /= len(data_loader)
    return metric_logger

# Evaluation function
@torch.no_grad()
def evaluate(model, data_loader, device):
    """Evaluate the model on the validation set"""
    model.eval()
    metric_logger = {}
    metric_logger['loss'] = 0
    
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Get model predictions
        outputs = model(images)
        
        # Compute metrics
        # This is a simplified version - in a real implementation,
        # you would compute AP, AR, and other relevant metrics
        
    return metric_logger

# Main training loop
def train_model(model, optimizer, data_loader_train, data_loader_val, device, num_epochs=10, 
               lr_scheduler=None, save_dir='models'):
    """Train a model with the given parameters"""
    # Create directory to save models
    os.makedirs(save_dir, exist_ok=True)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'lr': []
    }
    
    # Initial evaluation
    print("Initial evaluation...")
    eval_metrics = evaluate(model, data_loader_val, device)
    
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 20)
        
        # Train for one epoch
        metrics = train_one_epoch(model, optimizer, data_loader_train, 
                                 device, epoch, print_freq=10)
        
        # Update learning rate
        if lr_scheduler is not None:
            lr_scheduler.step()
        
        # Evaluate on validation set
        eval_metrics = evaluate(model, data_loader_val, device)
        
        # Save metrics
        history['train_loss'].append(metrics['loss'])
        history['val_loss'].append(eval_metrics.get('loss', 0))
        history['lr'].append(metrics['lr'])
        
        # Save model checkpoint
        checkpoint_path = os.path.join(save_dir, f"model_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history
        }, checkpoint_path)
        print(f"Checkpoint saved to {checkpoint_path}")
    
    # Save final model
    final_model_path = os.path.join(save_dir, "model_final.pth")
    torch.save(model.state_dict(), final_model_path)
    print(f"Final model saved to {final_model_path}")
    
    return history, model

# Configure training parameters for different models
def get_training_params(model_key):
    """Get recommended training parameters for a specific model"""
    common_params = {
        "num_epochs": 10,
        "batch_size": 4,
        "learning_rate": 0.001,
        "weight_decay": 0.0005,
    }
    
    # Model-specific parameters
    if model_key == "faster_rcnn_resnet50":
        return {**common_params, "learning_rate": 0.0001}
    elif model_key == "retinanet_resnet50":
        return {**common_params, "learning_rate": 0.00005}
    elif model_key == "ssdlite_mobilenet":
        return {**common_params, "learning_rate": 0.001, "batch_size": 16}
    elif model_key == "mask_rcnn_resnet50":
        return {**common_params, "learning_rate": 0.0001}
    else:
        return common_params

In [ ]:
# Full training function for any model
def train_tb_detection_model(model_key, dataset_config, training_config):
    """
    Complete function to train a tuberculosis detection model
    
    Args:
        model_key: Key identifying which model to train (e.g., 'faster_rcnn_resnet50')
        dataset_config: Dictionary containing dataset configuration
        training_config: Dictionary containing training parameters
        
    Returns:
        trained_model: The trained model
        history: Training history (losses, metrics)
    """
    print(f"Starting training for {model_key}...")
    print(f"Dataset config: {dataset_config}")
    print(f"Training config: {training_config}")
    
    # 1. Create the model
    model, model_name = create_model(model_key)
    print(f"Created {model_name} model")

    # 2. Create optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    if training_config.get("optimizer") == "adam":
        optimizer = torch.optim.Adam(
            params,
            lr=training_config.get("learning_rate", 0.001),
            weight_decay=training_config.get("weight_decay", 0.0005)
        )
    else:
        # Default to SGD
        optimizer = torch.optim.SGD(
            params,
            lr=training_config.get("learning_rate", 0.001),
            momentum=training_config.get("momentum", 0.9),
            weight_decay=training_config.get("weight_decay", 0.0005)
        )

    # 3. Create learning rate scheduler
    if training_config.get("scheduler") == "cosine":
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=training_config.get("num_epochs", 10)
        )
    else:
        # Default to StepLR
        lr_scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=training_config.get("step_size", 3),
            gamma=training_config.get("gamma", 0.1)
        )

    # 4. Get data loaders
    data_loader_train, data_loader_val = get_data_loaders(
        ann_file_train=dataset_config.get("ann_file_train"),
        ann_file_val=dataset_config.get("ann_file_val"),
        img_prefix=dataset_config.get("img_prefix"),
        img_list_train=dataset_config.get("img_list_train"),
        img_list_val=dataset_config.get("img_list_val"),
        batch_size=training_config.get("batch_size", 4)
    )

    # 5. Create directory for this model
    model_save_dir = f"models/{model_key}"
    if not os.path.exists(model_save_dir):
        os.makedirs(model_save_dir)

    # 6. Train the model
    history, trained_model = train_model(
        model=model,
        optimizer=optimizer,
        data_loader_train=data_loader_train,
        data_loader_val=data_loader_val,
        device=device,
        num_epochs=training_config.get("num_epochs", 10),
        lr_scheduler=lr_scheduler,
        save_dir=model_save_dir
    )

    # 7. Save final trained model and configuration
    config_path = os.path.join(model_save_dir, "training_config.json")
    with open(config_path, 'w') as f:
        json.dump({
            "model_key": model_key,
            "training_config": training_config,
            "dataset_config": {k: str(v) for k, v in dataset_config.items()}
        }, f, indent=4)
    
    print(f"Model training complete. Results saved in {model_save_dir}")
    return trained_model, history

# Example of how to use the training function (code ready to use, not executed)
"""
# Step 1: Set up dataset configuration
dataset_config = {
    "ann_file_train": os.path.join(ANNOTATIONS_DIR, "train.json"),
    "ann_file_val": os.path.join(ANNOTATIONS_DIR, "val.json"),
    "img_prefix": IMAGES_DIR,
    "img_list_train": os.path.join(LISTS_DIR, "all_train.txt"),
    "img_list_val": os.path.join(LISTS_DIR, "all_val.txt")
}

# Step 2: Set up training configuration 
training_config = {
    "num_epochs": 20,
    "batch_size": 4,
    "learning_rate": 0.001,
    "weight_decay": 0.0005,
    "momentum": 0.9,
    "optimizer": "sgd",  # or "adam"
    "scheduler": "step", # or "cosine"
    "step_size": 5,
    "gamma": 0.1
}

# Step 3: Choose model to train
model_key = "faster_rcnn_resnet50"  # Options: "faster_rcnn_resnet50", "retinanet_resnet50", "ssdlite_mobilenet", "mask_rcnn_resnet50"

# Step 4: Train the model
trained_model, history = train_tb_detection_model(model_key, dataset_config, training_config)

# Step 5: Evaluate on validation set (after training)
data_loader_val = get_data_loaders(
    ann_file_train=dataset_config["ann_file_train"],  # We don't need this but the function requires it
    ann_file_val=dataset_config["ann_file_val"],
    img_prefix=dataset_config["img_prefix"],
    img_list_val=dataset_config["img_list_val"],
    batch_size=4
)[1]  # Get only the validation loader

# Run evaluation
metrics = evaluate_model(trained_model, data_loader_val, device)

# Visualize evaluation results
eval_df = visualize_evaluation(metrics, class_names)
print(eval_df)
"""

print("The above code provides a complete training pipeline.")
print("You can copy and modify it to train on your own infrastructure.")
print("Make sure to update dataset paths before running.")

## 6. Evaluation and Metrics

Let's implement comprehensive evaluation metrics for TB detection models.

In [ ]:
# Evaluation metrics for object detection
def compute_ap_metrics(pred_boxes, pred_labels, pred_scores, 
                     target_boxes, target_labels,
                     iou_threshold=0.5):
    """
    Compute Average Precision metrics for object detection.
    
    Args:
        pred_boxes: List of predicted bounding boxes (N, 4)
        pred_labels: List of predicted class labels (N)
        pred_scores: List of prediction confidence scores (N)
        target_boxes: List of ground truth bounding boxes (M, 4)
        target_labels: List of ground truth class labels (M)
        iou_threshold: IoU threshold for determining a positive detection
        
    Returns:
        Dictionary of metrics including AP for each class
    """
    # Initialize metrics dictionary
    metrics = {
        "precision": {},
        "recall": {},
        "ap": {},
        "f1": {},
        "ap50": 0.0,
        "map": 0.0
    }
    
    # Process each class
    unique_classes = torch.unique(torch.cat([target_labels, pred_labels]))
    
    for cls in unique_classes:
        # Get predictions and targets for this class
        cls_pred_mask = pred_labels == cls
        cls_target_mask = target_labels == cls
        
        # Skip if no predictions or targets for this class
        if not cls_pred_mask.any() or not cls_target_mask.any():
            metrics["precision"][cls.item()] = 0.0
            metrics["recall"][cls.item()] = 0.0
            metrics["ap"][cls.item()] = 0.0
            metrics["f1"][cls.item()] = 0.0
            continue
            
        # Get class-specific predictions and targets
        cls_pred_boxes = pred_boxes[cls_pred_mask]
        cls_pred_scores = pred_scores[cls_pred_mask]
        cls_target_boxes = target_boxes[cls_target_mask]
        
        # Sort predictions by confidence score (descending)
        score_sorted_idx = torch.argsort(cls_pred_scores, descending=True)
        cls_pred_boxes = cls_pred_boxes[score_sorted_idx]
        cls_pred_scores = cls_pred_scores[score_sorted_idx]
        
        # Compute IoU between all predictions and targets
        if len(cls_pred_boxes) > 0 and len(cls_target_boxes) > 0:
            iou_matrix = box_iou(cls_pred_boxes, cls_target_boxes)
            
            # Find matches (IoU > threshold)
            max_iou, max_idx = iou_matrix.max(dim=1)
            matches = max_iou > iou_threshold
            
            # Compute precision and recall
            tp = matches.sum().float()
            fp = (~matches).sum().float()
            fn = len(cls_target_boxes) - tp
            
            precision = tp / (tp + fp) if tp + fp > 0 else torch.tensor(0.0)
            recall = tp / (tp + fn) if tp + fn > 0 else torch.tensor(0.0)
            f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else torch.tensor(0.0)
            
            # This is a simplified AP calculation
            # Real AP calculation would involve computing the area under the precision-recall curve
            ap = precision * recall
            
            metrics["precision"][cls.item()] = precision.item()
            metrics["recall"][cls.item()] = recall.item()
            metrics["ap"][cls.item()] = ap.item()
            metrics["f1"][cls.item()] = f1.item()
        else:
            metrics["precision"][cls.item()] = 0.0
            metrics["recall"][cls.item()] = 0.0
            metrics["ap"][cls.item()] = 0.0
            metrics["f1"][cls.item()] = 0.0
    
    # Compute mAP
    if metrics["ap"]:
        metrics["map"] = sum(metrics["ap"].values()) / len(metrics["ap"])
        metrics["ap50"] = metrics["map"]  # In this simplified version, AP@50 = mAP
    
    return metrics

# More comprehensive evaluation
@torch.no_grad()
def evaluate_model(model, data_loader, device):
    """
    Comprehensive evaluation of a TB detection model.
    Returns a variety of metrics for detection performance.
    """
    model.eval()
    
    all_metrics = {
        "ap": {},
        "precision": {},
        "recall": {},
        "f1": {},
        "map": 0.0,
        "ap50": 0.0
    }
    
    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Get predictions
        outputs = model(images)
        
        # Compute metrics for each image
        for output, target in zip(outputs, targets):
            pred_boxes = output['boxes']
            pred_labels = output['labels']
            pred_scores = output['scores']
            
            target_boxes = target['boxes']
            target_labels = target['labels']
            
            # Compute metrics
            metrics = compute_ap_metrics(
                pred_boxes, pred_labels, pred_scores,
                target_boxes, target_labels
            )
            
            # Accumulate metrics
            for cls, ap in metrics["ap"].items():
                if cls not in all_metrics["ap"]:
                    all_metrics["ap"][cls] = []
                    all_metrics["precision"][cls] = []
                    all_metrics["recall"][cls] = []
                    all_metrics["f1"][cls] = []
                
                all_metrics["ap"][cls].append(ap)
                all_metrics["precision"][cls].append(metrics["precision"][cls])
                all_metrics["recall"][cls].append(metrics["recall"][cls])
                all_metrics["f1"][cls].append(metrics["f1"][cls])
    
    # Compute final averages
    for cls in all_metrics["ap"]:
        all_metrics["ap"][cls] = sum(all_metrics["ap"][cls]) / len(all_metrics["ap"][cls])
        all_metrics["precision"][cls] = sum(all_metrics["precision"][cls]) / len(all_metrics["precision"][cls])
        all_metrics["recall"][cls] = sum(all_metrics["recall"][cls]) / len(all_metrics["recall"][cls])
        all_metrics["f1"][cls] = sum(all_metrics["f1"][cls]) / len(all_metrics["f1"][cls])
    
    # Compute mAP
    if all_metrics["ap"]:
        all_metrics["map"] = sum(all_metrics["ap"].values()) / len(all_metrics["ap"])
        all_metrics["ap50"] = all_metrics["map"]  # In this simplified version
    
    return all_metrics

# Function to visualize evaluation results
def visualize_evaluation(metrics_dict, class_names):
    """
    Visualize evaluation metrics for each class and overall model performance.
    """
    # Create a DataFrame for easy visualization
    data = []
    for i, cls in enumerate(class_names):
        if i in metrics_dict["ap"]:
            data.append({
                "Class": cls,
                "AP": metrics_dict["ap"][i],
                "Precision": metrics_dict["precision"][i],
                "Recall": metrics_dict["recall"][i],
                "F1": metrics_dict["f1"][i]
            })
    
    df = pd.DataFrame(data)
    
    # Add overall mAP
    df.loc[len(df)] = {
        "Class": "Overall",
        "AP": metrics_dict["map"],
        "Precision": df["Precision"].mean(),
        "Recall": df["Recall"].mean(),
        "F1": df["F1"].mean()
    }
    
    # Plot metrics
    plt.figure(figsize=(12, 8))
    
    # Bar chart for AP by class
    plt.subplot(2, 1, 1)
    sns.barplot(x="Class", y="AP", data=df)
    plt.title("Average Precision by Class")
    plt.xticks(rotation=45)
    
    # Precision-Recall scatter
    plt.subplot(2, 2, 3)
    sns.scatterplot(x="Recall", y="Precision", hue="Class", size="AP", sizes=(50, 200), data=df)
    plt.title("Precision vs Recall by Class")
    
    # F1 scores
    plt.subplot(2, 2, 4)
    sns.barplot(x="Class", y="F1", data=df)
    plt.title("F1 Scores by Class")
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return df

## 7. Inference and Visualization

Let's implement functions to perform inference on new chest X-ray images and visualize the detection results.

In [ ]:
# Function to perform inference on a single image
@torch.no_grad()
def detect_tb(model, image_path, device, confidence_threshold=0.5):
    """
    Perform TB detection on a single chest X-ray image.
    
    Args:
        model: The detection model
        image_path: Path to the image file
        device: Device to run inference on (CPU or GPU)
        confidence_threshold: Threshold for keeping predictions
        
    Returns:
        Original image, boxes, labels, and scores
    """
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image_tensor = F.pil_to_tensor(image).float() / 255.0
    
    # Move to device and add batch dimension
    image_tensor = image_tensor.to(device)
    
    # Perform inference
    model.eval()
    prediction = model([image_tensor])
    
    # Extract predictions
    boxes = prediction[0]["boxes"].cpu()
    labels = prediction[0]["labels"].cpu()
    scores = prediction[0]["scores"].cpu()
    
    # Filter predictions by confidence
    mask = scores >= confidence_threshold
    boxes = boxes[mask]
    labels = labels[mask]
    scores = scores[mask]
    
    return image, boxes, labels, scores

# Function to visualize detection results
def visualize_detection(image, boxes, labels, scores, class_names, save_path=None):
    """
    Visualize detection results on an image.
    
    Args:
        image: PIL Image
        boxes: Detected bounding boxes
        labels: Detected class labels
        scores: Confidence scores for detections
        class_names: List of class names for mapping label indices
        save_path: Path to save the visualization image (optional)
    """
    # Convert image to numpy array
    image_np = np.array(image)
    
    # Create a copy for drawing
    image_with_boxes = image_np.copy()
    
    # Define colors for different classes
    colors = [
        (0, 255, 0),    # Healthy - Green
        (0, 255, 255),  # Sick & Non-TB - Yellow
        (0, 0, 255),    # Active TB - Red
        (255, 0, 0),    # Latent TB - Blue
        (255, 0, 255),  # Active & Latent TB - Purple
        (128, 128, 128) # Uncertain TB - Gray
    ]
    
    # Draw bounding boxes and labels
    for i, (box, label, score) in enumerate(zip(boxes, labels, scores)):
        # Convert box to integers
        x1, y1, x2, y2 = [int(coord) for coord in box.tolist()]
        
        # Get class name
        class_id = label.item()
        class_name = class_names[class_id] if class_id < len(class_names) else f"Unknown ({class_id})"
        
        # Choose color
        color = colors[class_id % len(colors)]
        
        # Draw the box
        cv2.rectangle(image_with_boxes, (x1, y1), (x2, y2), color, 2)
        
        # Add label text
        label_text = f"{class_name}: {score.item():.2f}"
        cv2.putText(image_with_boxes, label_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Convert back to RGB for displaying with matplotlib
    image_with_boxes = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)
    
    # Display the image with detections
    plt.figure(figsize=(12, 10))
    plt.imshow(image_with_boxes)
    plt.title(f"TB Detection Results - {len(boxes)} detections")
    plt.axis('off')
    
    # Save if a path is provided
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    
    plt.show()
    
    # Print detection summary
    print(f"Detection Summary: {len(boxes)} TB lesions found")
    for i, (label, score) in enumerate(zip(labels, scores)):
        class_id = label.item()
        class_name = class_names[class_id] if class_id < len(class_names) else f"Unknown ({class_id})"
        print(f"  {i+1}. {class_name}: {score.item():.4f}")

# Function to perform batch inference
@torch.no_grad()
def batch_inference(model, image_list, img_prefix, device, confidence_threshold=0.5):
    """
    Perform batch inference on multiple images.
    
    Args:
        model: The detection model
        image_list: List of image paths or path to a text file containing image paths
        img_prefix: Prefix path to add to each image path
        device: Device to run inference on
        confidence_threshold: Threshold for keeping predictions
        
    Returns:
        Dictionary of results for each image
    """
    # Parse image list if it's a file
    if isinstance(image_list, str) and os.path.isfile(image_list):
        with open(image_list, 'r') as f:
            images = [line.strip() for line in f.readlines()]
    else:
        images = image_list
    
    results = {}
    
    # Process each image
    for img_path in tqdm(images, desc="Processing images"):
        # Build the full path
        full_path = os.path.join(img_prefix, img_path)
        
        if os.path.exists(full_path):
            # Perform detection
            image, boxes, labels, scores = detect_tb(
                model, full_path, device, confidence_threshold
            )
            
            # Store results
            results[img_path] = {
                "boxes": boxes,
                "labels": labels,
                "scores": scores
            }
        else:
            print(f"Warning: Image not found at {full_path}")
    
    return results

In [ ]:
# Example of running inference on a sample image (commented out)
"""
# Load a trained model
model_key = "faster_rcnn_resnet50"
model_path = f"models/{model_key}/model_final.pth"

# Create the model and load weights
model, _ = create_model(model_key)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)

# Sample image path (update this to a real path)
sample_image_path = os.path.join(IMAGES_DIR, "tb/tb0005.png")

# Perform detection
image, boxes, labels, scores = detect_tb(
    model, sample_image_path, device, confidence_threshold=0.5
)

# Visualize the results
visualize_detection(image, boxes, labels, scores, class_names)
"""

print("To perform inference on a sample image, uncomment and run the code above.")
print("Make sure to update the model path and sample image path accordingly.")

In [ ]:
# Model Comparison Framework
class ModelComparer:
    """
    Framework for comparing multiple trained TB detection models
    """
    def __init__(self, models_dict, class_names):
        """
        Initialize with a dictionary of trained models
        
        Args:
            models_dict: Dictionary of {model_name: model} pairs
            class_names: List of class names for the dataset
        """
        self.models = models_dict
        self.class_names = class_names
        self.results = {}
        
    def evaluate_all(self, data_loader, device):
        """
        Evaluate all models on the same dataset
        
        Args:
            data_loader: DataLoader for evaluation
            device: Device to run evaluation on
            
        Returns:
            Dictionary of evaluation results by model
        """
        for name, model in self.models.items():
            print(f"Evaluating {name}...")
            model.to(device)
            metrics = evaluate_model(model, data_loader, device)
            self.results[name] = metrics
        
        return self.results
    
    def compare_metrics(self):
        """
        Compare metrics across all evaluated models
        
        Returns:
            DataFrame with model comparison
        """
        if not self.results:
            return "No evaluation results available. Run evaluate_all first."
        
        comparison = []
        
        # For each model
        for model_name, metrics in self.results.items():
            # Overall metrics
            row = {
                "Model": model_name,
                "mAP": metrics["map"],
                "AP@50": metrics["ap50"]
            }
            
            # Add class-specific AP
            for cls_idx, cls_name in enumerate(self.class_names):
                if cls_idx in metrics["ap"]:
                    row[f"AP_{cls_name}"] = metrics["ap"][cls_idx]
                else:
                    row[f"AP_{cls_name}"] = None
            
            comparison.append(row)
        
        return pd.DataFrame(comparison)
    
    def visualize_comparison(self):
        """Visualize model comparison results"""
        if not self.results:
            return "No evaluation results available. Run evaluate_all first."
        
        df = self.compare_metrics()
        
        # Plot mAP comparison
        plt.figure(figsize=(12, 6))
        sns.barplot(x="Model", y="mAP", data=df)
        plt.title("Mean Average Precision (mAP) by Model")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        # Plot per-class AP comparison
        ap_cols = [col for col in df.columns if col.startswith("AP_")]
        if ap_cols:
            plt.figure(figsize=(14, 8))
            
            # Reshape data for grouped bar chart
            plot_data = []
            for _, row in df.iterrows():
                for ap_col in ap_cols:
                    cls_name = ap_col[3:]  # Remove "AP_" prefix
                    plot_data.append({
                        "Model": row["Model"],
                        "Class": cls_name,
                        "AP": row[ap_col]
                    })
            
            plot_df = pd.DataFrame(plot_data)
            sns.barplot(x="Class", y="AP", hue="Model", data=plot_df)
            plt.title("Per-Class Average Precision by Model")
            plt.xticks(rotation=45)
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
        
        return df
    
    def inference_comparison(self, image_path, confidence_threshold=0.5, save_dir=None):
        """
        Compare inference results across models on a single image
        
        Args:
            image_path: Path to the image file
            confidence_threshold: Threshold for detections
            save_dir: Directory to save result images (optional)
        """
        if not self.models:
            return "No models available for comparison."
        
        # Load the image once
        image = Image.open(image_path).convert("RGB")
        
        # Create a figure with subplots
        n_models = len(self.models)
        cols = min(2, n_models)
        rows = (n_models + cols - 1) // cols
        
        plt.figure(figsize=(15, 5*rows))
        
        # Process each model
        for i, (name, model) in enumerate(self.models.items()):
            model.to(device)
            model.eval()
            
            # Prepare image tensor
            image_tensor = F.pil_to_tensor(image).float() / 255.0
            image_tensor = image_tensor.to(device)
            
            # Perform inference
            with torch.no_grad():
                prediction = model([image_tensor])
            
            # Extract predictions
            boxes = prediction[0]["boxes"].cpu()
            labels = prediction[0]["labels"].cpu()
            scores = prediction[0]["scores"].cpu()
            
            # Filter by confidence threshold
            mask = scores >= confidence_threshold
            boxes = boxes[mask]
            labels = labels[mask]
            scores = scores[mask]
            
            # Create subplot
            plt.subplot(rows, cols, i+1)
            
            # Draw detections on image
            image_np = np.array(image)
            image_with_boxes = image_np.copy()
            
            # Colors for different classes
            colors = [
                (0, 255, 0),    # Healthy - Green
                (0, 255, 255),  # Sick & Non-TB - Yellow
                (0, 0, 255),    # Active TB - Red
                (255, 0, 0),    # Latent TB - Blue
                (255, 0, 255),  # Active & Latent TB - Purple
                (128, 128, 128) # Uncertain TB - Gray
            ]
            
            # Draw boxes and labels
            for j, (box, label, score) in enumerate(zip(boxes, labels, scores)):
                x1, y1, x2, y2 = [int(coord) for coord in box.tolist()]
                class_id = label.item()
                class_name = self.class_names[class_id] if class_id < len(self.class_names) else f"Unknown ({class_id})"
                color = colors[class_id % len(colors)]
                
                # Draw box and label
                cv2.rectangle(image_with_boxes, (x1, y1), (x2, y2), color, 2)
                label_text = f"{class_name}: {score.item():.2f}"
                cv2.putText(image_with_boxes, label_text, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            # Convert back to RGB and display
            image_with_boxes = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)
            plt.imshow(image_with_boxes)
            plt.title(f"{name}: {len(boxes)} detections")
            plt.axis('off')
            
            # Save individual result if requested
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)
                save_path = os.path.join(save_dir, f"{name}_detection.png")
                plt.imsave(save_path, image_with_boxes)
        
        plt.tight_layout()
        
        # Save combined figure if requested
        if save_dir:
            plt.savefig(os.path.join(save_dir, "model_comparison.png"))
        
        plt.show()


# Example of how to use the model comparison framework
"""
# Step 1: Set up model paths and load trained models
model_paths = {
    "faster_rcnn_resnet50": "models/faster_rcnn_resnet50/model_final.pth",
    "retinanet_resnet50": "models/retinanet_resnet50/model_final.pth",
    "ssdlite_mobilenet": "models/ssdlite_mobilenet/model_final.pth"
}

# Step 2: Load models
trained_models = {}
for model_key, path in model_paths.items():
    if os.path.exists(path):
        model, model_name = create_model(model_key)
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        trained_models[model_name] = model
        print(f"Loaded {model_name}")
    else:
        print(f"Model not found at {path}")

# Step 3: Set up model comparer
comparer = ModelComparer(trained_models, class_names)

# Step 4: Create validation data loader for comparison
val_data_loader = get_data_loaders(
    ann_file_train=os.path.join(ANNOTATIONS_DIR, "train.json"),
    ann_file_val=os.path.join(ANNOTATIONS_DIR, "val.json"),
    img_prefix=IMAGES_DIR,
    img_list_val=os.path.join(LISTS_DIR, "all_val.txt"),
    batch_size=4
)[1]  # Get only validation loader

# Step 5: Evaluate all models
results = comparer.evaluate_all(val_data_loader, device)

# Step 6: Compare metrics and visualize
comparison_df = comparer.visualize_comparison()
print(comparison_df)

# Step 7: Compare detections on a sample image
sample_image_path = os.path.join(IMAGES_DIR, "tb/tb0005.png")
comparer.inference_comparison(sample_image_path, confidence_threshold=0.5, save_dir="comparison_results")
"""

print("Model comparison framework defined. Use it to compare different trained models.")
print("This code allows you to systematically evaluate and compare detection performance.")